# Skript fürs Training eines RL-Agenten zur Auftragssteuerung
## Hier: Schritt 1, Lager: Trainiere Lager unabhängig vom AGV (der Feinschliff kann ein Training abgestimmt aufs AGV sein)

Der state wird für das Lager anders definiert als für das AGV, möglichst viel soll OHE sein:
10:  Erste 10 Einträge: Anzahl an zu liefernden Waren an die Stationen
10: Nächste 10 Einträge Ampelsystem für Dringlichkeit, Stufe 0: noch nicht dringend, Stufe 1: wird bald dringend, Stufe 2: dringend, Stufe 3: schon zu spät, jetzt aber hallo ;-)
10: Dann 10 Einträge, welche Waren schon auf dem AGV sind
10: Dann 10 Einträge als OHE-Maske, welche Stationen sinnvoll sind anzufahren (nach heuristischer Methodik: Nur die Stationen, die offene Aufträge haben).

Aktion:
1: Fertig, schicke weg von Lager
10: Lade Waren für Station "action" aufs AGV

In [1]:
import csv
import random
import numpy as np
##### Settting parameters ####

#### AGV
V_MAX = 20
AGV_MAX_CAP = 4

#### ENV
ORDER_TIME_BONUS = 300
STATION_PROCESSING_TIME = 20
STATION_COUNT = 13
TIME_FACTOR = 400


### Points
ORDER_OVERDUE_MINUS = - 0.2 
ORDER_OVERDUE_MINUS_EVERY_SECONDS = 20
POINTS_ORDER_FULFILLED = 1


### Rewards
#REWARD_ORDER_OVERDUE = -0.01


#### Schritt 1: Verteile großen Malus für unsinnige Aufträge, alle anderen 0
#### Schritt 2: Verteile für die anderen auch sinnige Belohnungen
TIME_URGENT_RED = 320
TIME_URGENT_ORANGE = 640
TIME_URGENT_YELLOW = 960


In [2]:
import factory_environment as env
import agv
import logic
import game_mechanics_4_storage_training as gm_4_st
import RL_Agent

In [3]:
my_env = env.env('ttable.csv', 'station_number.csv', 'env_control_order.csv')

In [4]:
my_agv = agv.agv()

In [5]:
my_logic = logic.logic()

In [6]:
### State size
# agv: Ladezustand +  ENV: order_list 
STATION_COUNT = 13
# Ladezustand + act_stat + ENV: order list
state_size_agv = 4+13+4+13
state_size_storage = 4*10 + 1
#state_size = state_size_storage

#### Action size
# Fahre zu einer der 13 Stationen
action_size_agv = 13
### Lade auf für eine der 11 Stationen
action_size_storage = 11

In [7]:
gm = gm_4_st.game_mechanics(my_logic, my_env, my_agv)


## ⚙️ Reinforcement Learning Agent Parameters Explained

### 1. **gamma (γ) – Discount Factor**
- **Purpose**: Controls how much the agent values future rewards versus immediate ones.
- **Range**: Between 0 and 1.
- **Example**:
  - `γ = 0.99` → agent cares a lot about long-term rewards.
  - `γ = 0.1` → agent focuses mostly on immediate results.
- **Effect**: Higher values encourage farsightedness; lower values make the agent more short-term reactive.

---

### 2. **epsilon (ε) – Exploration Rate**
- **Purpose**: Determines the probability of the agent choosing a **random action** (exploration) versus the **best-known action** (exploitation).
- **Range**: Between 0 and 1.
- **Behavior**:
  - `ε = 1.0` → pure exploration (tries everything).
  - `ε = 0.01` → mostly exploits learned policy.
- **Why It Matters**: Without exploration, the agent might miss better strategies.

---

### 3. **epsilon_min**
- **Purpose**: Minimum bound for `epsilon` during training.
- **Prevents**: The agent from becoming completely deterministic too early.
- **Example**: `epsilon_min = 0.01` → agent will always have a 1% chance of trying something new.

---

### 4. **epsilon_decay**
- **Purpose**: Determines how quickly the `epsilon` value decays over time.
- **Example**:
  - `epsilon *= 0.995` per episode → slow decay.
  - `epsilon *= 0.9` → faster decay.
- **Goal**: Start explorative and gradually become confident in its knowledge.

---

### 5. **learning_rate (α)**
- **Purpose**: Determines how much new knowledge overrides old Q-values during updates.
- **Range**: Small positive float, like `0.001`.
- **Trade-off**:
  - High value: Learns fast, might oscillate or forget.
  - Low value: Stable but slow learning.

---

### 6. **memory (Replay Buffer)**
- **Type**: `deque` (double-ended queue) storing past experiences.
- **Why**: Helps decorrelate experiences for better training stability.
- **Stored Values**: Each memory is a tuple of `(state, action, reward, next_state, done)`.

---

### 7. **batch_size**
- **Purpose**: Number of experiences randomly sampled from memory to train on each time.
- **Effect**:
  - Small batch → more frequent updates, less stable.
  - Large batch → more stable updates, slower cycles.

---

### 8. **state_size**
- **What**: Number of features that describe the current environment state.
- **Use**: Defines the input shape of your neural network.

### 9. **action_size**
- **What**: Number of possible actions the agent can take.
- **Use**: Defines the output shape of your neural network.

---

Let me know if you'd like a cheat sheet summarizing this, or if you’re curious about how these interact during training. We can also dive deeper into things like target networks or dueling architectures when you're ready to advance! 🧠✨

In [8]:
START_POS = 1   # Startet an Lager B

### Sonstige Training-Variablen
GAMES_PER_EPISODE = 8
EPISODE_COUNT = 250



#### Variablen für RL-Agent
GAMMA = 0.5  #0.99
EPSILON = 1#0.25#0.99995
EPSILON_MIN = 0.01
EPSILON_DECAY = 0.997
LEARNING_RATE = 0.001#*10
BATCH_SIZE = 512


In [ ]:
loading_storage = ''

my_RL_agent_storage = RL_Agent.DQNAgent(len(gm.act_state_storage), action_size_storage, GAMMA, EPSILON, EPSILON_MIN, EPSILON_DECAY, LEARNING_RATE, loading_storage)
my_RL_agent_storage.model.summary()

c:\Reinforcement Learning\.venv\Lib\site-packages\keras\src\saving\saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 10 variables whereas the saved optimizer has 18 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 41)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │         5,376 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 11)             │           363 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 32,152 (125.60 KB)

 Trainable params: 16,075 (62.79 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 16,077 (62.80 KB)

In [10]:
######
reward_vec = []
total_time_vec = []
gm.reset()
for episode in range(EPISODE_COUNT):
    for game in range(GAMES_PER_EPISODE):
        total_reward, total_time = gm.run_game(my_RL_agent_storage)
        reward_vec.append(total_reward)
        total_time_vec.append(total_time)
        gm.reset()
        my_env.reset()
        my_agv.reset()
    my_RL_agent_storage.replay(BATCH_SIZE)
    print(f'epsilon was: {my_RL_agent_storage.epsilon}')
    my_RL_agent_storage.epsilon *=EPSILON_DECAY
    print(f'mean reward episode {episode}: {np.mean(reward_vec[:-GAMES_PER_EPISODE+1])}')
    print(f'mean total time: {np.mean(total_time_vec[:-GAMES_PER_EPISODE+1])}')
    
        

epsilon was: 1
mean reward episode 0: 2.75
mean total time: 697.4509803921569
epsilon was: 0.997
mean reward episode 1: 2.2377777777777776
mean total time: 638.1584680655887
epsilon was: 0.994009
mean reward episode 2: 2.3194117647058823
mean total time: 722.935510633562
epsilon was: 0.991026973
mean reward episode 3: -2.800799999999989
mean total time: 999.7293429652563
epsilon was: 0.985089730404757
mean reward episode 4: -1.5254545454545376
mean total time: 929.5701076815627


KeyboardInterrupt: 

In [ ]:
print(f'mittlere Belohnung für die ersten 100 Spiele: {np.mean(reward_vec[0:100])}')
print(f'mittlere Belohnung für die letzten 100 Spiele: {np.mean(reward_vec[:-100])}')
print(f'Abschließendes epsilon: {my_RL_agent_storage.epsilon}')

mittlere Belohnung für die ersten 100 Spiele: -0.09779999999999558
mittlere Belohnung für die letzten 100 Spiele: -3.94718260869564
Abschließendes epsilon: 0.19512707759140924


In [ ]:
mean_reward_per_episode = []
for i in range(0,len(reward_vec),GAMES_PER_EPISODE):
    if i % GAMES_PER_EPISODE == 0:
        mean_reward_per_episode.append(np.mean(reward_vec[i:i+GAMES_PER_EPISODE]))
#mean_reward_per_episode        

In [ ]:
#my_RL_agent_storage.memory

In [ ]:
my_RL_agent_storage.model.save('./model_storage.h5')

In [ ]:
#my_RL_agent_storage.model.save('./model_storage.keras')

In [ ]:
import os

base_path = './models/storage_V'
version = 1
while os.path.exists(f'{base_path}{version}.keras'):
    version += 1
my_RL_agent_storage.model.save(f'{base_path}{version}.keras')